In [6]:
import cornac
from cornac.data import Reader
from cornac.datasets import movielens
from cornac.data import Dataset, FeatureModality
from cornac.eval_methods import RatioSplit, StratifiedSplit
from cornac.metrics import RMSE, AUC, NDCG, Precision, Recall
from cornac.models import MF, ItemKNN, UserKNN, NMF, BPR, LightGCN, SVD
import pandas as pd
import numpy as np
import random
import math
from collections import OrderedDict
import seaborn as sns
import matplotlib.pyplot as plt

/Users/tahsinalamgirkheya/anaconda3/envs/cornac/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# movie_data = reader.read(fpath="./data/indexed_movies.csv", sep=",", fmt="UIRT")
# movie_data
movies = pd.read_csv("./cornac/data_c/indexed_movies.csv")

movies = movies.drop(columns=movies.columns[0])
movies[:4]

# unique_genres = set("|".join(movies["genres"]).split("|"))
# unique_genres = list(unique_genres)
unique_genres = [
    "Action",
    "Thriller",
    "Romance",
    "Western",
    "Children's",
    "Mystery",
    "Fantasy",
    "Film-Noir",
    "Documentary",
    "Comedy",
    "Adventure",
    "Sci-Fi",
    "Horror",
    "Crime",
    "Musical",
    "War",
    "Animation",
    "Drama",
]
for genre in unique_genres:
    movies[genre] = 0
for index, row in movies.iterrows():
    genres = row["genres"].split("|")
    for genre in genres:
        movies.at[index, genre] = 1

# item_categories = movies[unique_genres]
# unique_iids = rating_data_pd['itemID'].unique()
# movies = movies[movies['itemID'].isin(unique_iids)]
genre = movies[unique_genres]
item_features_numpy = genre.to_numpy()
# print(item_features_numpy.shape)

# item_categories = item_categor
item_features = {
    str(item_id): {"genre_" + str(idx): value for idx, value in enumerate(row)}
    for item_id, row in enumerate(item_features_numpy)
}
ids = list(range(0, 3416))
# item_feature_modality = FeatureModality(
#     features=item_features_numpy, ids=ids, normalized=True
# )


users = pd.read_csv("./cornac/data_c/u_id_mapping.csv", sep="\t")
users = users.drop(columns=users.columns[0])
gender_map = {"M": 0, "F": 1}
users["Gender"] = users["Gender"].map(gender_map)
# unique_uids = rating_data_pd['userID'].unique()

# users = users[users["userID"].isin(unique_uids)]
user_features_numpy = users.to_numpy()
print(user_features_numpy.shape)
print(item_features_numpy.shape)
# user_feature_modality = FeatureModality(
#     features=user_features_numpy, name="user", normalized=True, ids=list(range(0, 6040))
# )

# print("Example Item Features:")
# for item_id, features in list(item_features.items())[:5]:
#     print(f"Item ID: {item_id}, Features: {features}")

(6040, 2)
(3416, 18)


In [10]:
user_ids = users.to_numpy()[:, 1]
item_ids = movies.to_numpy()[:, 2]
user_ids.__len__()

6040

In [13]:
reco_matrix = np.load("reco_matrix.npy")
reco_matrix_mapped_items = np.load("reco_matrix_mapped_items.npy")
reco_matrix_mapped_scores = np.load("reco_matrix_mapped_scores.npy")
reco_matrix_all = np.load("reco_matrix_all.npy")
reco_items_scores_all = np.load("reco_items_scores_all.npy")

In [15]:
reco_matrix_mapped_scores[1][1][120]
# reco_matrix_mapped_items[1][1][120]
# np.where(reco_matrix_all[1][1]==267)[0][0]
# np.where(reco_matrix_mapped_items[1][1]==120)[0][0]
# reco_items_scores_all[1][1136]
# x=  reco_items_scores_all
# x
# np.save("reco_items_scores_all.npy",reco_items_scores_all)

3.8498077934799873

In [17]:
# reco_matrix_mapped_scores[1][1][2278]
# reco_items_scores_all[1][1136]
score_dicts[1][267]

3.8498077934799873

In [111]:


# # reco_matrix_all_items
score_dicts[1][1136]

3.950089035183898

In [16]:
score_dicts = []
for i in range(reco_matrix_all[1].shape[0]):
    iids = reco_matrix_all[1][i]
    score = reco_items_scores_all[i]
    score_dicts.append(OrderedDict(zip(iids, score)))

In [ ]:
# top_k = 20
# reco_matrix_n = np.zeros((len(user_ids), top_k), dtype=int)

# for i in range(reco_matrix[0].shape[0]):
#     reco_matrix_n[i] = reco_matrix[1][i][:top_k]

# reco_matrix_n.shape

In [191]:
from cornac.reranking.Calibration import Calibration

config = {"user_genre_dist_file": "./cornac/data_c/user_genre_dist_new.csv"}
c = Calibration(
    config=config, movies=movies, top_k=50, unique_genres=unique_genres, users=users
)
# for itemknn

reranked_reco = c.get_improved_reco(reco_matrix[1], reco_matrix_all[1], score_dicts)

['Action', 'Thriller', 'Romance', 'Western', "Children's", 'Mystery', 'Fantasy', 'Film-Noir', 'Documentary', 'Comedy', 'Adventure', 'Sci-Fi', 'Horror', 'Crime', 'Musical', 'War', 'Animation', 'Drama']
[1767]
[1767, 25]
[1767, 25, 1562]
[1767, 25, 1562, 611]
[1767, 25, 1562, 611, 2316]
[1767, 25, 1562, 611, 2316, 681]
[1767, 25, 1562, 611, 2316, 681, 696]
[1767, 25, 1562, 611, 2316, 681, 696, 3244]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102, 162]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102, 162, 408]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102, 162, 408, 298]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102, 162, 408, 298, 101]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102, 162, 408, 298, 101, 1884]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102, 162, 408, 298, 101, 1884, 906]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102, 162, 408, 298, 101, 1884, 906, 1322]
[1767, 25, 1562, 611, 2316, 681, 696, 3244, 2102, 162, 